# Quick start: JetClass

The second week (T4 to T7) runs the analysis of the first week on JetClass, the data set of the Particle Transformer
paper, and looks inside the pretrained ParT. This notebook reads the three JetClass validation files of the course
(QCD, W and top), shows that they have the same format as the toy files, recomputes their observables, puts a toy
jet next to a JetClass jet and runs the pretrained ParT with its attention and integrated gradients.

It needs the files of `jetgen fetch jetclass --split val --classes QCD,Wqq,Tbqq` and `jetgen fetch models ParT_kin`
(or `python examples/make_task_data.py --week 2 --only fetch`). Without them it falls back to the 150 JetClass jets
committed in `tests/data/` and skips the pretrained model. JetClass, ParT and weaver are described in
[docs/jetclass.md](../docs/jetclass.md); the tasks are T4 and T5 of [TASKS.md](../TASKS.md).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

import itj
import itj.jetclass
from itj import plot
from itj.features import labels
from itj.interpret import integrated_gradients
from itj.io import load
from itj.metrics import accuracy, auc, one_vs_one, roc
from itj.models import load_pretrained, make_inputs
from itj.observables import nsubjettiness, softdrop
from itj.toy import generate
from itj.train import predict

print("itj", itj.__version__, "| torch", torch.__version__)
ROOT = Path("..")  # the repository root, seen from notebooks/

## The files

The validation file of each class holds 100 000 jets. They are read with the split of the course, per file: here the
first 2000 jets of each test part.

In [ ]:
try:
    files = itj.jetclass.files("val", root=ROOT / "data" / "jetclass")
    data = load(files, part="test", max_jets=2000)
except FileNotFoundError as error:
    print(error)
    print("Using the 150 JetClass jets of tests/data instead.")
    data = load(ROOT / "tests" / "data" / "jetclass_sample.parquet")
data

The branches, their types and their order are those of the toy files, without the truth columns (`file_index` says
which file a jet comes from). The labels are one-hot branches; `labels` turns them into the class index of the three
classes of the course (0 QCD, 1 W, 2 top).

In [ ]:
print(len(data.names), "branches:", ", ".join(data.names[:6]), "...")
print("label_Wqq:", data["label_Wqq"].dtype, "| classes:", np.bincount(labels(data, "jets3")))

## One jet

`part_deta` is not simply $\eta_i - \eta_J$: JetClass flips its sign for jets at negative $\eta$, so that a positive
`part_deta` always points towards larger $|\eta|$, the forward direction. The displays use `part_deta` and
`part_dphi`.

In [ ]:
names = data.process_names
w = int(np.flatnonzero(names == "wqq")[0])
top = int(np.flatnonzero(names == "tbqq")[0])
jet = data.jet(w)
p = jet.particles
eta = np.arcsinh(p["part_pz"] / np.hypot(p["part_px"], p["part_py"]))
sign = np.sign(jet.jet["jet_eta"])
print("jet_eta", round(float(jet.jet["jet_eta"]), 3))
print(
    "part_deta == (eta - jet_eta) * sign(jet_eta):",
    np.allclose(p["part_deta"], (eta - jet.jet["jet_eta"]) * sign, atol=1e-3),
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), constrained_layout=True)
plot.jet_display(data, w, ax=axes[0])
plot.jet_display(data, top, ax=axes[1])
plt.show()

## Observables

The stored `jet_sdmass` and `jet_tau1` to `jet_tau3` are recomputed from the particles: soft drop with
$z_\mathrm{cut} = 0.1$, $\beta = 0$ and $\tau_N$ with $\beta = 1$. JetClass (Delphes) finds the $\tau_N$ axes
with one-pass minimised exclusive-$k_t$ axes; the WTA axes of the toy files give nearly the same values
([observables.md](../docs/observables.md)).

In [ ]:
p4, mask = data.padded_p4()
sd = softdrop(p4, mask)
stored = np.asarray(data["jet_sdmass"], dtype=np.float64)
print(f"m_SD within 1 % of jet_sdmass: {np.mean(np.abs(sd.mass - stored) <= 0.01 * stored):.1%} of the jets")
for axes in ("wta_kt", "onepass_kt"):
    tau = nsubjettiness(p4, mask, n=(1, 2, 3), axes=axes)
    ratios = []
    for k in range(3):
        ref = np.asarray(data[f"jet_tau{k + 1}"], dtype=np.float64)
        ok = ref > 0
        ratios.append(np.median(tau[ok, k] / ref[ok]))
    print(f"{axes:<11s} median tau_N / jet_tauN:", np.round(ratios, 3))

## A toy jet next to a JetClass jet

A W jet of the toy at the detector level and the JetClass W jet above, coloured by particle type. Task T4.1 compares
the two data sets on many jets.

In [ ]:
toy = generate(20, processes=("wqq",), seed=4)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), constrained_layout=True)
plot.jet_display(toy, 0, color="type", ax=axes[0], title="toy W jet")
plot.jet_display(data, w, color="type", ax=axes[1], title="JetClass W jet")
plt.show()
print("particles: toy", int(toy["jet_nparticles"][0]), "| JetClass", int(data["jet_nparticles"][w]))

## The pretrained ParT

`load_pretrained` reads the checkpoint of the ParT paper (`ParT_kin`, ten classes, 2.1 M parameters). On the CPU it
runs at about 200 jets per second, so we take 2000 jets; `probs_to_jets3` keeps the QCD, Wqq and Tbqq outputs and renormalises them.

In [ ]:
pretrained = Path(os.environ.get("ITJ_PRETRAINED_DIR", ROOT / "models" / "pretrained"))
model = None
if (pretrained / "ParT_kin.pt").exists():
    model = load_pretrained("ParT_kin", root=pretrained)
    print(model.name, f"{model.n_parameters():,} parameters")
else:
    print(f"no checkpoint in {pretrained}: run `jetgen fetch models ParT_kin`; the cells below are skipped")

In [ ]:
if model is not None:
    sub = data.select(np.arange(len(data))[:: max(1, len(data) // 2000)][:2000])
    probs = itj.jetclass.probs_to_jets3(predict(model, sub, batch_size=256))
    y = labels(sub, "jets3")
    print(f"{len(sub)} jets, accuracy (3 classes) {accuracy(probs, y):.3f}")
    curves = {}
    for name, signal in (("W vs QCD", 1), ("top vs QCD", 2)):
        ys, ss, _ = one_vs_one(probs, y, signal, 0)
        curves[name] = roc(ys, ss)[:2]
        print(f"{name}: AUC {auc(ys, ss):.4f}")
    plot.roc_curves(curves)
    plt.show()

## What the class token looks at

`trace(batch, attention=True)` returns every intermediate output of the model. The class attention of the first
class-attention block (averaged over its eight heads) gives a weight to every particle of the top jet: drawn on its
display, it can be compared with the three prongs of the decay (task T5.3 tests such pictures).

In [ ]:
if model is not None:
    one = data.select([top])
    batch = make_inputs(model, one)
    with torch.no_grad():
        trace = model.trace(batch, attention=True)
    attention = trace.class_attention["cls0"][0, :, 0, 1:].mean(0).numpy()  # entry 0 is the class token itself
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), constrained_layout=True)
    plot.jet_display(one, 0, ax=axes[0], title="top jet")
    plot.jet_display(one, 0, color=attention, label="attention weight", ax=axes[1], title="class attention, cls0")
    plt.show()

## Integrated gradients

Integrated gradients attribute the top score of the same jet to its particles (32 steps from an all-zero input,
signed). The completeness gap measures how well the attributions add up to the change of the score.

In [ ]:
if model is not None:
    attributions, gap = integrated_gradients(model, batch, target=8, steps=32)  # 8 = Tbqq among the ten classes
    print("completeness gap:", np.round(gap, 4))
    fig, ax = plt.subplots(figsize=(5.2, 4.4), constrained_layout=True)
    plot.attribution_display(one, 0, attributions[0], ax=ax)
    plt.show()

## Next steps

- Task T4 of [TASKS.md](../TASKS.md): the toy against JetClass, the stored branches, the pretrained taggers, training on
  JetClass on a CPU, the transfer between the toy and JetClass, and weaver
  ([examples/weaver](../examples/weaver/README.md)).
- Task T5: attributions, attention, the pair bias $U$ and IRC safety, each checked on the toy before JetClass.